# Experimento A, segunda versão: os 3 métodos na mesma prova
Este notebook refaz a avaliação das imputações do experimento A (notebook `imputacoes_outliers_tratados`), na base tratada, com duas ideias do experimento B (SIICUSP):
1. **Mesma prova.** Em cada rodada e taxa de falhas (5, 10, 20 e 30%), um único sorteio remove valores de todas as variáveis, e a média, a mediana e o KNN imputam as mesmas células. No A, a média e a mediana eram testadas com 5% de falhas e o KNN com 30%: os métodos não eram comparados nas mesmas células.
2. **KNN padronizado.** As variáveis passam a ter média 0 e desvio padrão 1 antes da busca pelos vizinhos, para que as de escala grande (ex.: Cond.) não dominem a distância entre os meses. O k e os pesos são escolhidos pelo erro, em rodadas separadas das da avaliação. No A, o KNN usava k = 2, pesos iguais e as variáveis na unidade original.

O A original continua no notebook dele; a última seção compara os dois desenhos nas células que eles têm em comum.

# Set Up

In [1]:
import pandas as pd
from pcj.utils import dados, BASE_DIR
from pcj.metricas_avaliacao import avaliacao_mesma_prova, metricas_juntando

In [2]:
df, metadados, variaveis = dados(r"data\processed\dados_med_tratados.xlsx")

# Escolha do k do KNN
Grade de k (1, 3, 5 … 15) e de pesos (`uniform`: pesos iguais; `distance`: os vizinhos mais próximos pesam mais), a mesma do `knn_regressor`. Cada combinação roda só o KNN nas seeds 100 a 129, que sorteiam outras células, não as da avaliação (seeds 0 a 29): escolher o k com as células da prova deixaria o resultado otimista. O critério é o R² médio entre as 16 variáveis (o R² de cada variável junta as 30 rodadas); fica a combinação com a maior média entre as 4 taxas.

In [3]:
grade_k = [1, 3, 5, 7, 9, 11, 13, 15]
grade_pesos = ["uniform", "distance"]
linhas = []

for k in grade_k:
    for pesos in grade_pesos:
        # Só o KNN, nas seeds 100 a 129 (outras células, não as da avaliação).
        # O _ recebe a tabela por rodada, que aqui não é usada
        _, celulas_k = avaliacao_mesma_prova(df, seeds=range(100, 130), metodos=("knn",),
                                             n_neighbors=k, weights=pesos, devolver_celulas=True)
        juntando_k = metricas_juntando(celulas_k)

        # R² médio entre as variáveis, em cada taxa (cada grupo = as 16 variáveis de uma taxa)
        for taxa, grupo in juntando_k.groupby("pct_faltantes"):
            linhas.append({
                "k": k,
                "pesos": pesos,
                "pct_faltantes": taxa,
                "r2_medio": grupo["r2"].mean()
            })

escolha_k = pd.DataFrame(linhas)

Uma linha por k e pesos; uma coluna por taxa de falhas (R² médio entre as variáveis) e a média das 4 taxas.

In [4]:
# pivot: uma linha por (k, pesos), uma coluna por taxa
tabela_escolha = escolha_k.pivot(index=["k", "pesos"], columns="pct_faltantes", values="r2_medio")

# Média das 4 taxas (mean(axis=1) tira a média de cada linha)
tabela_escolha["media_taxas"] = tabela_escolha.mean(axis=1)

# idxmax devolve o rótulo da linha com o maior valor: aqui, o par (k, pesos)
k_escolhido, pesos_escolhidos = tabela_escolha["media_taxas"].idxmax()
print("Escolhido (maior R² médio entre as taxas): k =", k_escolhido, "| pesos =", pesos_escolhidos)

# O melhor de cada taxa, para ver se a escolha mudaria com a taxa
for taxa in [0.05, 0.1, 0.2, 0.3]:
    k_taxa, pesos_taxa = tabela_escolha[taxa].idxmax()
    print(f"  melhor a {taxa:.0%}: k = {k_taxa}, pesos = {pesos_taxa}")

tabela_escolha.round(3)

Escolhido (maior R² médio entre as taxas): k = 13 | pesos = distance
  melhor a 5%: k = 13, pesos = distance
  melhor a 10%: k = 13, pesos = distance
  melhor a 20%: k = 13, pesos = distance
  melhor a 30%: k = 13, pesos = distance


pct_faltantes   0.05    0.1    0.2    0.3  media_taxas
k  pesos                                              
1  distance   -0.078 -0.093 -0.167 -0.268       -0.152
   uniform    -0.078 -0.093 -0.167 -0.268       -0.152
3  distance    0.285  0.254  0.188  0.145        0.218
   uniform     0.275  0.239  0.177  0.143        0.208
5  distance    0.313  0.304  0.250  0.216        0.271
   uniform     0.297  0.286  0.233  0.209        0.256
7  distance    0.340  0.321  0.277  0.243        0.295
   uniform     0.322  0.301  0.259  0.231        0.278
9  distance    0.335  0.327  0.288  0.254        0.301
   uniform     0.314  0.305  0.267  0.238        0.281
11 distance    0.338  0.330  0.291  0.260        0.305
   uniform     0.317  0.307  0.267  0.241        0.283
13 distance    0.344  0.331  0.292  0.261        0.307
   uniform     0.323  0.306  0.266  0.239        0.284
15 distance    0.342  0.328  0.291  0.259        0.305
   uniform     0.320  0.302  0.263  0.234        0.280

# Avaliação
30 rodadas (seeds 0 a 29) × 4 taxas, com os 3 métodos nas mesmas células e o KNN padronizado, com o k e os pesos escolhidos.

In [5]:
# tabela: uma linha por seed × taxa × método × variável (as métricas de cada rodada)
# celulas: uma linha por célula avaliada (valor verdadeiro e imputado), usada para juntar as rodadas
tabela, celulas = avaliacao_mesma_prova(df, n_neighbors=int(k_escolhido), weights=pesos_escolhidos,
                                        devolver_celulas=True)
tabela.head()

,variavel,rmse,mae,r2,n_celulas,metodo,pct_faltantes,seed
0,COR,69.890878,64.212389,-0.005868,5,media,0.05,0
1,TURB.,53.223806,45.874336,-0.002340,5,media,0.05,0
2,pH,0.150585,0.122478,-1.180371,5,media,0.05,0
3,ALC.,14.586900,14.263717,-0.996787,5,media,0.05,0
4,AC.,3.942513,3.113274,-0.407918,5,media,0.05,0


Conferência: meses de referência, número de rodadas e células avaliadas por variável em cada taxa (as mesmas nos 3 métodos)

In [6]:
print("Meses completos (referência):", len(df.dropna()))
print("Rodadas:", tabela["seed"].nunique())

# Células por variável em cada rodada: o número é o mesmo em todas as rodadas e métodos,
# por isso basta a primeira linha de cada método e taxa (first)
tabela.groupby(["pct_faltantes", "metodo"], sort=False)["n_celulas"].first().unstack()

Meses completos (referência): 118
Rodadas: 30


metodo,media,mediana,knn
pct_faltantes,,,
0.05,5,5,5
0.10,11,11,11
0.20,23,23,23
0.30,35,35,35


# Métricas juntando as rodadas
Como no A: RMSE, MAE e R² calculados uma vez sobre todas as células de cada método × taxa × variável (`metricas_juntando`), e o DP do RMSE e do MAE entre as 30 rodadas.

In [7]:
juntando = metricas_juntando(celulas)

# DP entre as rodadas: o groupby separa a tabela por método, taxa e variável; o std calcula o DP das 30 rodadas
desvio = tabela.groupby(["metodo", "pct_faltantes", "variavel"], sort=False)[["rmse", "mae"]].std()

# add_suffix renomeia as colunas (rmse -> rmse_dp, mae -> mae_dp); reset_index devolve as chaves do groupby às colunas
desvio = desvio.add_suffix("_dp").reset_index()

# merge junta as duas tabelas pelas linhas com o mesmo método, a mesma taxa e a mesma variável
resumo = juntando.merge(desvio, on=["metodo", "pct_faltantes", "variavel"])
resumo = resumo[["metodo", "pct_faltantes", "variavel", "rmse", "rmse_dp", "mae", "mae_dp",
                 "r2", "n_celulas", "n_seeds"]]
print("Linhas:", len(resumo), "(3 métodos × 4 taxas × 16 variáveis)")
resumo.head()

Linhas: 192 (3 métodos × 4 taxas × 16 variáveis)


,metodo,pct_faltantes,variavel,rmse,rmse_dp,mae,mae_dp,r2,n_celulas,n_seeds
0,media,0.05,COR,85.262182,32.729295,67.398820,23.477550,-0.019924,150,30
1,media,0.05,TURB.,70.938156,24.046574,56.683068,16.965179,-0.016209,150,30
2,media,0.05,pH,0.153302,0.052803,0.122301,0.044843,-0.027537,150,30
3,media,0.05,ALC.,18.989151,5.387964,16.134277,4.822825,-0.026134,150,30
4,media,0.05,AC.,5.191109,2.428368,3.643304,1.731324,-0.018567,150,30


## Visão geral
R² médio entre as 16 variáveis, por método e taxa, e em quantas das 16 variáveis o KNN teve RMSE menor que a média e que a mediana (nas mesmas células).

In [8]:
# pivot_table com aggfunc="mean": média do R² das 16 variáveis, para cada método (linhas) e taxa (colunas)
visao_geral = resumo.pivot_table(index="metodo", columns="pct_faltantes", values="r2", aggfunc="mean")
visao_geral = visao_geral.loc[["media", "mediana", "knn"]]
print("R² médio entre as variáveis")
display(visao_geral.round(3))

linhas = []
for taxa in [0.05, 0.1, 0.2, 0.3]:
    # Linhas desta taxa, com (método, variável) como índice, para comparar os métodos variável a variável
    resumo_taxa = resumo[resumo["pct_faltantes"] == taxa].set_index(["metodo", "variavel"])
    rmse_knn = resumo_taxa.loc["knn", "rmse"]
    linhas.append({
        "pct_faltantes": taxa,
        "knn_menor_que_media": (rmse_knn < resumo_taxa.loc["media", "rmse"]).sum(),
        "knn_menor_que_mediana": (rmse_knn < resumo_taxa.loc["mediana", "rmse"]).sum()
    })

print("Variáveis (de 16) em que o KNN teve RMSE menor")
display(pd.DataFrame(linhas))

R² médio entre as variáveis


pct_faltantes,0.05,0.10,0.20,0.30
metodo,,,,
media,-0.024,-0.019,-0.019,-0.022
mediana,-0.099,-0.091,-0.097,-0.099
knn,0.407,0.349,0.310,0.264


Variáveis (de 16) em que o KNN teve RMSE menor


,pct_faltantes,knn_menor_que_media,knn_menor_que_mediana
0,0.05,15,16
1,0.10,15,16
2,0.20,15,16
3,0.30,16,16


## Por variável
Uma tabela por métrica: variáveis nas linhas; uma coluna para cada método e taxa.

In [9]:
for metrica in ["rmse", "mae", "r2"]:
    # pivot: variáveis nas linhas; colunas em dois níveis (método, taxa)
    tabela_metrica = resumo.pivot(index="variavel", columns=["metodo", "pct_faltantes"], values=metrica)

    # Variáveis na ordem da base e métodos na ordem média, mediana, KNN
    tabela_metrica = tabela_metrica.loc[variaveis, ["media", "mediana", "knn"]]

    print(metrica.upper())
    display(tabela_metrica.round(3))

RMSE


metodo             media                                     mediana  \
pct_faltantes       0.05       0.10       0.20       0.30       0.05   
variavel                                                               
COR               85.262    108.269    101.399    103.984     92.787   
TURB.             70.938     66.148     69.652     70.691     80.094   
pH                 0.153      0.150      0.158      0.157      0.158   
ALC.              18.989     19.724     19.473     19.304     18.799   
AC.                5.191      5.519      5.246      5.117      5.497   
O.C.               1.830      2.312      2.291      2.322      1.831   
O.D.               0.822      0.835      0.816      0.837      0.821   
Cl                13.829     13.397     13.429     13.632     13.781   
DUR.              14.833     14.547     14.514     14.782     15.195   
Fe                 1.655      1.831      1.858      1.877      1.768   
Mn                 0.096      0.089      0.088      0.084      0.099   
Cond.            135.864    128.379    131.301    132.048    136.203   
Cianobacteria  17071.380  14823.258  13846.812  15518.466  18022.927   
C.F.           54295.695  49399.131  48736.061  48549.529  56907.947   
Clorofila         26.697     28.778     27.881     28.691     27.786   
F                  0.121      0.126      0.119      0.117      0.122   

metodo                                                knn             \
pct_faltantes       0.10       0.20       0.30       0.05       0.10   
variavel                                                               
COR              115.793    108.844    112.089     44.595     68.412   
TURB.             72.559     76.816     79.191     45.572     43.776   
pH                 0.154      0.164      0.164      0.129      0.127   
ALC.              19.670     19.679     19.464      9.588     11.453   
AC.                5.800      5.515      5.315      4.228      5.191   
O.C.               2.317      2.308      2.336      1.622      2.211   
O.D.               0.832      0.821      0.835      0.649      0.662   
Cl                13.359     13.454     13.717      8.567      8.734   
DUR.              14.750     14.745     15.013     11.091     11.252   
Fe                 1.979      1.999      2.007      1.012      1.183   
Mn                 0.092      0.092      0.088      0.085      0.080   
Cond.            129.856    134.716    134.540     63.924     63.826   
Cianobacteria  15654.926  14385.019  16195.707  15753.750  13871.990   
C.F.           51680.633  51334.187  50755.442  55686.967  50274.456   
Clorofila         30.624     29.255     30.330     22.845     24.236   
F                  0.126      0.119      0.117      0.092      0.111   

metodo                               
pct_faltantes       0.20       0.30  
variavel                             
COR               66.719     75.612  
TURB.             49.371     53.675  
pH                 0.137      0.142  
ALC.              11.888     12.870  
AC.                5.051      4.945  
O.C.               2.193      2.283  
O.D.               0.682      0.719  
Cl                 9.344     10.009  
DUR.              11.567     12.268  
Fe                 1.257      1.320  
Mn                 0.083      0.080  
Cond.             71.371     77.318  
Cianobacteria  13205.161  15110.063  
C.F.           48944.961  48388.858  
Clorofila         24.382     25.959  
F                  0.106      0.106

MAE


metodo             media                                     mediana  \
pct_faltantes       0.05       0.10       0.20       0.30       0.05   
variavel                                                               
COR               67.399     75.500     72.919     73.639     63.867   
TURB.             56.683     54.128     55.428     55.776     53.620   
pH                 0.122      0.120      0.127      0.126      0.119   
ALC.              16.134     16.220     15.851     15.593     15.580   
AC.                3.643      3.736      3.564      3.476      3.460   
O.C.               1.467      1.686      1.636      1.666      1.465   
O.D.               0.653      0.666      0.652      0.668      0.649   
Cl                10.441     10.252     10.345     10.385     10.187   
DUR.              12.103     12.100     11.998     12.032     12.213   
Fe                 1.330      1.437      1.437      1.425      1.219   
Mn                 0.064      0.062      0.061      0.060      0.058   
Cond.            109.426    102.777    106.196    107.744    105.933   
Cianobacteria  11003.927   9729.397   9325.954   9682.758   9596.113   
C.F.           26386.267  25773.255  26363.634  26933.090  20780.007   
Clorofila         21.116     21.976     21.455     21.726     19.222   
F                  0.090      0.092      0.088      0.086      0.091   

metodo                                                knn             \
pct_faltantes       0.10       0.20       0.30       0.05       0.10   
variavel                                                               
COR               69.736     66.797     68.420     30.361     34.341   
TURB.             48.261     49.948     51.994     29.698     28.225   
pH                 0.119      0.127      0.129      0.099      0.096   
ALC.              15.855     15.732     15.458      7.299      8.511   
AC.                3.503      3.381      3.245      2.888      3.337   
O.C.               1.693      1.655      1.681      1.327      1.561   
O.D.               0.661      0.658      0.666      0.492      0.507   
Cl                 9.991     10.196     10.286      5.902      6.103   
DUR.              12.048     11.996     12.040      8.055      8.480   
Fe                 1.338      1.336      1.302      0.661      0.785   
Mn                 0.057      0.056      0.055      0.053      0.053   
Cond.            101.955    107.151    107.570     46.268     47.042   
Cianobacteria   8484.333   7809.946   8393.203   8477.340   7914.232   
C.F.           19302.785  20161.243  20085.854  24982.572  24462.617   
Clorofila         20.943     20.094     20.615     15.642     16.277   
F                  0.092      0.088      0.086      0.066      0.076   

metodo                               
pct_faltantes       0.20       0.30  
variavel                             
COR               36.217     41.629  
TURB.             32.480     35.777  
pH                 0.107      0.110  
ALC.               8.775      9.441  
AC.                3.357      3.305  
O.C.               1.506      1.584  
O.D.               0.510      0.548  
Cl                 6.591      7.113  
DUR.               8.598      9.143  
Fe                 0.822      0.845  
Mn                 0.054      0.053  
Cond.             52.539     57.681  
Cianobacteria   7678.142   8401.482  
C.F.           24811.529  25059.221  
Clorofila         16.919     17.879  
F                  0.072      0.073

R2


metodo         media                      mediana                         knn  \
pct_faltantes   0.05   0.10   0.20   0.30    0.05   0.10   0.20   0.30   0.05   
variavel                                                                        
COR           -0.020 -0.017 -0.021 -0.028  -0.208 -0.163 -0.176 -0.194  0.721   
TURB.         -0.016 -0.015 -0.014 -0.025  -0.295 -0.221 -0.233 -0.286  0.581   
pH            -0.028 -0.021 -0.016 -0.027  -0.087 -0.070 -0.099 -0.120  0.271   
ALC.          -0.026 -0.019 -0.013 -0.023  -0.006 -0.014 -0.035 -0.040  0.738   
AC.           -0.019 -0.021 -0.021 -0.026  -0.142 -0.128 -0.128 -0.107  0.324   
O.C.          -0.030 -0.017 -0.017 -0.019  -0.031 -0.021 -0.033 -0.031  0.191   
O.D.          -0.043 -0.026 -0.029 -0.028  -0.043 -0.019 -0.040 -0.022  0.349   
Cl            -0.019 -0.020 -0.017 -0.011  -0.012 -0.014 -0.021 -0.024  0.609   
DUR.          -0.019 -0.017 -0.020 -0.027  -0.069 -0.046 -0.053 -0.059  0.430   
Fe            -0.027 -0.029 -0.025 -0.017  -0.172 -0.203 -0.186 -0.162  0.616   
Mn            -0.022 -0.015 -0.014 -0.014  -0.099 -0.099 -0.101 -0.092  0.190   
Cond.         -0.018 -0.022 -0.027 -0.023  -0.023 -0.046 -0.082 -0.062  0.775   
Cianobacteria -0.023 -0.017 -0.020 -0.016  -0.140 -0.135 -0.101 -0.106  0.129   
C.F.          -0.012 -0.016 -0.016 -0.028  -0.112 -0.112 -0.128 -0.124 -0.065   
Clorofila     -0.031 -0.014 -0.016 -0.017  -0.116 -0.148 -0.119 -0.136  0.245   
F             -0.027 -0.019 -0.015 -0.017  -0.032 -0.016 -0.015 -0.014  0.412   

metodo                              
pct_faltantes   0.10   0.20   0.30  
variavel                            
COR            0.594  0.558  0.457  
TURB.          0.555  0.491  0.409  
pH             0.272  0.234  0.170  
ALC.           0.656  0.622  0.545  
AC.            0.097  0.054  0.042  
O.C.           0.071  0.067  0.016  
O.D.           0.355  0.282  0.242  
Cl             0.566  0.507  0.455  
DUR.           0.391  0.352  0.293  
Fe             0.571  0.531  0.497  
Mn             0.170  0.111  0.101  
Cond.          0.747  0.696  0.649  
Cianobacteria  0.109  0.072  0.037  
C.F.          -0.052 -0.025 -0.021  
Clorofila      0.281  0.223  0.168  
F              0.204  0.189  0.173

# Comparação com o A original
Com as mesmas seeds, as células sorteadas a 5% são as mesmas da média e da mediana do A, e as sorteadas a 30% são as mesmas do KNN do A. Os resultados do A vêm do Excel gravado pelo notebook `imputacoes_outliers_tratados` (se algo mudar lá, rode aquele antes deste).
- Média e mediana a 5%: os dois desenhos imputam igual, e a diferença precisa ser 0 (conferência).
- KNN a 30%: o A usava k = 2, pesos iguais e a unidade original; aqui, as variáveis padronizadas e o k escolhido. A diferença (novo − A) mostra o efeito da mudança: negativa no RMSE e no MAE e positiva no R² = o KNN novo foi melhor.

In [10]:
caminho_A = BASE_DIR / "data" / "output" / "imputacoes_outliers_tratados.xlsx"
resumo_A = pd.read_excel(caminho_A, sheet_name="resumo")

# merge junta as linhas do A e do novo com o mesmo método, a mesma taxa e a mesma variável.
# suffixes diferencia as colunas de mesmo nome nas duas tabelas (rmse_A, rmse_novo...)
comparacao_A = resumo_A.merge(resumo, on=["metodo", "pct_faltantes", "variavel"], suffixes=("_A", "_novo"))
for metrica in ["rmse", "mae", "r2"]:
    comparacao_A[f"{metrica}_diferenca"] = comparacao_A[f"{metrica}_novo"] - comparacao_A[f"{metrica}_A"]

comparacao_A = comparacao_A[["metodo", "pct_faltantes", "variavel",
                             "rmse_A", "rmse_novo", "rmse_diferenca",
                             "mae_A", "mae_novo", "mae_diferenca",
                             "r2_A", "r2_novo", "r2_diferenca"]]

# Conferência: média e mediana a 5% (a maior diferença, em valor absoluto, precisa ser ~0;
# só sobra o arredondamento da gravação no Excel)
media_mediana = comparacao_A[comparacao_A["metodo"].isin(["media", "mediana"])]
print("Média e mediana a 5%: linhas", len(media_mediana), "| maior diferença absoluta:",
      media_mediana[["rmse_diferenca", "mae_diferenca", "r2_diferenca"]].abs().max().max())

Média e mediana a 5%: linhas 32 | maior diferença absoluta: 7.275957614183426e-12


KNN a 30%: A (k = 2, pesos iguais, unidade original) × novo (padronizado, com o k e os pesos escolhidos)

In [11]:
knn_30 = comparacao_A[comparacao_A["metodo"] == "knn"].set_index("variavel")
knn_30.drop(columns=["metodo", "pct_faltantes"]).round(3)

,rmse_A,rmse_novo,rmse_diferenca,mae_A,mae_novo,mae_diferenca,r2_A,r2_novo,r2_diferenca
variavel,,,,,,,,,
COR,107.922,75.612,-32.309,67.733,41.629,-26.104,-0.107,0.457,0.564
TURB.,71.656,53.675,-17.982,49.486,35.777,-13.709,-0.053,0.409,0.462
pH,0.175,0.142,-0.033,0.138,0.110,-0.028,-0.267,0.170,0.436
ALC.,17.920,12.870,-5.051,13.525,9.441,-4.084,0.118,0.545,0.427
AC.,6.165,4.945,-1.220,4.370,3.305,-1.065,-0.489,0.042,0.531
O.C.,2.746,2.283,-0.463,1.960,1.584,-0.377,-0.424,0.016,0.440
O.D.,0.904,0.719,-0.185,0.716,0.548,-0.168,-0.200,0.242,0.442
Cl,13.522,10.009,-3.513,9.921,7.113,-2.809,0.005,0.455,0.450
DUR.,14.883,12.268,-2.615,11.362,9.143,-2.219,-0.041,0.293,0.334


# Gravação
Abas `escolha_k` (R² médio por k, pesos e taxa), `resumo` (métricas juntando + DP do RMSE e do MAE), `rodadas` (uma linha por seed × taxa × método × variável) e `comparacao_A`. Rodar de novo dá os mesmos números: as seeds são fixas, e o KNN não tem sorteio.

In [12]:
caminho = BASE_DIR / "data" / "output" / "imputacoes_mesma_prova.xlsx"

# ExcelWriter permite gravar várias abas no mesmo arquivo
with pd.ExcelWriter(caminho) as arquivo:
    escolha_k.to_excel(arquivo, sheet_name="escolha_k", index=False)
    resumo.to_excel(arquivo, sheet_name="resumo", index=False)
    tabela.to_excel(arquivo, sheet_name="rodadas", index=False)
    comparacao_A.to_excel(arquivo, sheet_name="comparacao_A", index=False)

print("Gravado em:", caminho)

Gravado em: C:\_Master\Programacao\Guilherme-Betta\1_Projects\IC2\IC2-item-4\data\output\imputacoes_mesma_prova.xlsx
